# 02 — Data Cleaning

**Objective:** Clean the raw World Cup Players data, apply meaningful transformations, and export a processed file to `data/processed/`.

**Input:** `data/raw/Primary/WorldCupPlayers.csv` (37,784 rows × 9 columns)  
**Output:** `data/processed/cleaned_worldcup_players.csv`

## 2.1 — Setup & Imports

In [ ]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

RAW_PATH = PROJECT_ROOT / "data/raw/Primary/WorldCupPlayers.csv"
PROCESSED_PATH = PROJECT_ROOT / "data/processed/cleaned_worldcup_players.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw input    : {RAW_PATH}")
print(f"Clean output : {PROCESSED_PATH}")

## 2.2 — Load Raw Data

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 2.3 — Step 1: Standardize Column Names (snake_case)

In [ ]:
df = df_raw.copy()

# Convert to snake_case
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Renamed columns:")
for old, new in zip(df_raw.columns, df.columns):
    marker = " ←" if old != new else ""
    print(f"  {old:20s} → {new}{marker}")

## 2.4 — Step 2: Strip Whitespace from String Columns

In [ ]:
str_cols = df.select_dtypes(include="object").columns.tolist()
print(f"String columns to strip: {str_cols}")

for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

print("Whitespace stripped.")

## 2.5 — Step 3: Remove Exact Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after : {after:,}")
print(f"Duplicates removed: {before - after:,}")

## 2.6 — Step 4: Handle Shirt Number = 0 (Unknown)

In [ ]:
zero_count = (df["shirt_number"] == 0).sum()
print(f"Rows with shirt_number == 0: {zero_count:,} ({zero_count / len(df) * 100:.1f}%)")
print("Interpretation: Early tournaments did not use numbered jerseys.")

# Replace 0 with NaN to distinguish from actual shirt numbers
df["shirt_number"] = df["shirt_number"].replace(0, pd.NA)
print(f"\nShirt number nulls after fix: {df['shirt_number'].isna().sum():,}")

## 2.7 — Step 5: Clean Position Column

In [ ]:
print("Position distribution BEFORE cleaning:")
print(df["position"].value_counts(dropna=False))
print()

# GK  = Goalkeeper
# C   = Captain (outfield player — captain status tracked via is_captain)
# GKC = Goalkeeper + Captain
# NaN = Outfield (non-captain, non-GK)

# Create two clear boolean flags (stored as int 0/1):
df["is_goalkeeper"] = df["position"].isin(["GK", "GKC"]).astype(int)
df["is_captain"] = df["position"].isin(["C", "GKC"]).astype(int)

# Position reflects playing role, not captaincy:
# C → Outfield (captain status is in is_captain)
# GK → Goalkeeper
# GKC → GK-Captain
# NaN → Outfield
df["position"] = df["position"].fillna("Outfield").replace({
    "GK": "Goalkeeper",
    "C": "Outfield",
    "GKC": "GK-Captain"
})

print("Position distribution AFTER cleaning:")
print(df["position"].value_counts())
print(f"\nis_goalkeeper == 1 count: {df['is_goalkeeper'].sum():,}")
print(f"is_captain == 1 count  : {df['is_captain'].sum():,}")

## 2.8 — Step 6: Map Line-up Values

In [ ]:
print("Line-up distribution BEFORE:")
print(df["line_up"].value_counts())
print()

df["line_up"] = df["line_up"].replace({"S": "Starting", "N": "Substitute"})

print("Line-up distribution AFTER:")
print(df["line_up"].value_counts())

## 2.9 — Step 7: Parse Event Column

The `event` column encodes match events as coded strings like `G40' Y72' I80'`.  
We'll parse these into structured numeric/boolean columns.

**Event codes handled:**

| Code | Meaning | Parsed Into |
|------|---------|-------------|
| `G` | Goal | `goals` (count) |
| `P` | Penalty goal | `penalties_scored` (count) |
| `W` | Own goal | `own_goals` (count) |
| `Y` | Yellow card | `yellow_cards` (count, adjusted for RSY) |
| `R` | Red card | `red_cards` (count) |
| `RSY` | Red via 2nd yellow | `second_yellow_reds` (count) |
| `MP` | Missed penalty | `missed_penalties` (count) |
| `I` / `IH` | Substitution in / in at half | `sub_in` (0/1) |
| `O` / `OH` | Substitution out / out at half | `sub_out` (0/1) |

**Codes identified but not parsed (rare/contextual):**
- `ETO` — Extra-time only marker (not a player event)
- `N` — No-show / administrative notation (not a match event)

In [ ]:
def count_event(event_str, code):
    """Count occurrences of an event code (e.g., 'G', 'Y') in the event string."""
    if pd.isna(event_str) or event_str == "<NA>":
        return 0
    # Match standalone event code followed by digits and apostrophe
    # Use negative lookbehind to avoid matching inside compound codes
    pattern = rf"(?<![A-Z]){code}\d+\'?"
    return len(re.findall(pattern, str(event_str)))


def has_sub(event_str, direction):
    """Check for substitution events: I/IH (in) or O/OH (out).
    Handles both regular subs (I62') and half-time subs (IH, IH46').
    Returns int 0 or 1.
    """
    if pd.isna(event_str) or event_str == "<NA>":
        return 0
    # Match direction code, optional H (half-time), optional digits+apostrophe
    pattern = rf"(?<![A-Z]){direction}H?\d*\'?"
    return 1 if re.search(pattern, str(event_str)) else 0


# Parse events into new columns
df["goals"] = df["event"].apply(lambda x: count_event(x, "G"))
df["penalties_scored"] = df["event"].apply(lambda x: count_event(x, "P"))
df["own_goals"] = df["event"].apply(lambda x: count_event(x, "W"))
df["yellow_cards"] = df["event"].apply(lambda x: count_event(x, "Y"))
df["red_cards"] = df["event"].apply(lambda x: count_event(x, "R"))
df["second_yellow_reds"] = df["event"].apply(lambda x: count_event(x, "RSY"))
df["missed_penalties"] = df["event"].apply(lambda x: count_event(x, "MP"))
df["sub_in"] = df["event"].apply(lambda x: has_sub(x, "I"))
df["sub_out"] = df["event"].apply(lambda x: has_sub(x, "O"))

# Fix double-counting: RSY includes a yellow that is already counted in Y;
# subtract so yellow_cards reflects only standalone yellows.
df["yellow_cards"] = df["yellow_cards"] - df["second_yellow_reds"]

# Derived column: total dismissals (straight red + second-yellow red)
df["total_dismissals"] = df["red_cards"] + df["second_yellow_reds"]

print("Parsed event columns summary:")
event_cols = ["goals", "penalties_scored", "own_goals", "yellow_cards",
              "red_cards", "second_yellow_reds", "total_dismissals",
              "missed_penalties", "sub_in", "sub_out"]
for col in event_cols:
    print(f"  {col:22s} Total: {df[col].sum():>5,}  |  Max per row: {df[col].max()}")

## 2.10 — Step 8: Extract Coach Nationality

In [ ]:
def extract_coach_nationality(coach_str):
    """Extract the 3-letter nationality code from 'SURNAME Name (NAT)' format."""
    if pd.isna(coach_str):
        return pd.NA
    match = re.search(r"\(([A-Z]+)\)", str(coach_str))
    return match.group(1) if match else pd.NA


df["coach_nationality"] = df["coach_name"].apply(extract_coach_nationality)

# Quick check: how many coaches manage a team different from their nationality?
foreign_coaches = df[df["team_initials"] != df["coach_nationality"]]
total_rows = len(df)
foreign_pct = len(foreign_coaches) / total_rows * 100

print(f"Coach nationality extracted successfully.")
print(f"Foreign coach rows: {len(foreign_coaches):,} / {total_rows:,} ({foreign_pct:.1f}%)")
print(f"\nTop 10 coach nationalities:")
print(df["coach_nationality"].value_counts().head(10))

## 2.11 — Cleaning Summary & Final Schema

In [ ]:
print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"Raw rows             : {before:>10,}")
print(f"Duplicates removed   : {before - after:>10,}")
print(f"Clean rows           : {after:>10,}")
print(f"Columns (raw)        : {len(df_raw.columns):>10}")
print(f"Columns (clean)      : {len(df.columns):>10}")
print(f"New columns added    : {len(df.columns) - len(df_raw.columns):>10}")
print("=" * 60)
print("\nFinal column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col} ({df[col].dtype})")

In [ ]:
# Null check on final dataframe
print("Null counts in cleaned dataframe:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls (except intentional NaN in shirt_number).")
print(f"\nshirt_number NaN count: {df['shirt_number'].isna().sum():,} (intentional — old tournaments)")

In [ ]:
df.head(10)

## 2.12 — Export Cleaned Dataset

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)

# Verify
verify = pd.read_csv(PROCESSED_PATH)
print(f"Saved cleaned dataset to: {PROCESSED_PATH}")
print(f"Verified shape: {verify.shape[0]:,} rows × {verify.shape[1]} columns")

## Cleaning Decisions Log

| Step | Action | Rationale |
|------|--------|-----------|
| 1 | Snake_case column names | Consistency, Pythonic convention |
| 2 | Strip whitespace | Remove invisible trailing/leading spaces |
| 3 | Drop 736 exact duplicates | Data quality — identical rows carry no new info |
| 4 | Shirt Number 0 → NaN | Zero means "unknown" in pre-numbering era, not jersey #0 |
| 5 | Position: NaN → Outfield, C → Outfield, GK → Goalkeeper, GKC → GK-Captain | Captain status tracked separately in `is_captain` |
| 6 | Line-up: S → Starting, N → Substitute | Human-readable labels |
| 7 | Parse Event into 10 structured columns (incl. `total_dismissals`) | Enable quantitative analysis of goals, cards, subs |
| 7a | Include IH / OH half-time substitution codes in sub_in / sub_out | ~600 half-time sub events were previously missed |
| 7b | Subtract `second_yellow_reds` from `yellow_cards` | Avoid double-counting the yellow that triggers RSY |
| 7c | Cast `is_goalkeeper`, `is_captain`, `sub_in`, `sub_out` → int (0/1) | Ensures consistent numeric export in CSV |
| 8 | Extract coach nationality from Coach Name | Enables foreign-coach analysis |

### Event Codes — Full Inventory

| Code | Parsed? | Column | Notes |
|------|---------|--------|-------|
| G | ✅ | `goals` | Goal scored |
| P | ✅ | `penalties_scored` | Penalty goal |
| W | ✅ | `own_goals` | Own goal |
| Y | ✅ | `yellow_cards` | Yellow card (adjusted for RSY) |
| R | ✅ | `red_cards` | Straight red card |
| RSY | ✅ | `second_yellow_reds` | Red via second yellow |
| MP | ✅ | `missed_penalties` | Missed penalty |
| I | ✅ | `sub_in` | Substituted in |
| IH | ✅ | `sub_in` | Substituted in at half-time |
| O | ✅ | `sub_out` | Substituted out |
| OH | ✅ | `sub_out` | Substituted out at half-time |
| ETO | ❌ | — | Extra-time only marker; contextual, not a player action |
| N | ❌ | — | Administrative notation; not a match event |

✅ **Proceed to Notebook 03 (EDA).**